In [1]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from itertools import combinations
from itertools import product

from sklearn.utils.parallel import Parallel, delayed

%run Model_functions.ipynb

In [2]:
BASE_DIR = "../../DATA/AGB_DATA/Merged_Data/"
#BASE_DIR = "../../Data/"

In [3]:
def filter_existing(feat_list, available):
    """Keep only features that survived filtering. Warn about dropped ones."""
    kept    = [f for f in feat_list if f in available]
    dropped = [f for f in feat_list if f not in available]
    if dropped:
        print(f"  Dropped (not in dataframe): {dropped}")
    return kept

In [4]:
def create_interact_terms(X, interact_col_names):
    for col in interact_col_names:
        if col in X.columns:
            X[f'height_x_{col}'] = X['height'] * X[col]

    return X

In [5]:
def generate_feature_combinations(feature_groups, categ_cols, required_groups=None):
    if required_groups is None:
        required_groups = []

    combos = []
    group_names = list(feature_groups.keys())

    for r in range(1, len(group_names) + 1):
        for selected_groups in combinations(group_names, r):

            # Skip useless combos
            if all(g in categ_cols for g in selected_groups):
                continue
            
            features = []
            for group in selected_groups:
                features.extend(feature_groups[group])

            features = list(dict.fromkeys(features))

            combos.append({
                "group_combo": selected_groups,
                "features": features,
                "num_features": len(features)
            })

    return combos

# SENTINEL + CANOPY DATA

In [6]:
SENTINEL_DATA_CSV        = BASE_DIR + "/Sentinel_AGB/AGB_SENTINEL_CANOPY.csv"
sentinel_raw_df = pd.read_csv(SENTINEL_DATA_CSV)

assert len(sentinel_raw_df["simard_height_m"].head())
assert len(sentinel_raw_df["tandemx_height_m"].head())

In [7]:
sentinel_raw_df.columns

Index(['dataset', 'plot_id', 'start_date', 'end_date', 'latitude', 'longitude',
       'diameter', 'height', 'species', 'plant_AGB_kg', 'capture_start',
       'capture_end', 'sentinel_time', 'Blue', 'Green', 'Red', 'RE1', 'RE2',
       'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1',
       'NDRE2', 'NDRE3', 'CIrededge', 'CLOUD_COVERAGE', 'cloud_threshold_used',
       'simard_height_m', 'tandemx_height_m'],
      dtype='object')

## Sentinel feature selection

In [8]:
sent_non_feature_cols = [
    'plant_AGB_kg',        # Target variable
    'dataset',             # metadata
    'sentinel_time',       # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
    'diameter',            # Allometric
    'height',               # Allometric
    'cloud_threshold_used',
    'start_date'
]

SENT_interact_cols = [
    # Indices (excluding MNDWI which isn't a biomass index)
    'EVI', 'NBR', 'NDVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge',
    # Raw structural bands
    'NIR', 'SWIR1', 'SWIR2',
]

target = 'plant_AGB_kg'
y_sent = sentinel_raw_df[target]

sent_feature_cols = [c for c in sentinel_raw_df.columns if c not in sent_non_feature_cols]

### Feature selection with Simard canopy height

In [9]:
X_sent_t = sentinel_raw_df[sent_feature_cols].copy()

# Select TANDEMX
X_sent_t = X_sent_t.rename({'tandemx_height_m': 'height'}, axis=1)
X_sent_t = X_sent_t.drop(columns=['simard_height_m'])

### Create interaction terms

In [10]:
X_sent_t = create_interact_terms(X_sent_t, SENT_interact_cols)

### Feature selection with Simard canopy height

In [11]:
X_sent_s = sentinel_raw_df[sent_feature_cols].copy()

# Select SIMARD
X_sent_s = X_sent_s.rename({'simard_height_m': 'height'}, axis=1)
X_sent_s = X_sent_s.drop(columns=['tandemx_height_m'])

### Create interaction terms

In [12]:
X_sent_s = create_interact_terms(X_sent_s, SENT_interact_cols)

## Sentinel test features

In [13]:
test_cv = 5

In [14]:
sent_struct_features = ['height']
species              = ['species']

sent_bands   = ['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2']
sent_indices = ['NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge']

sent_top_spectral  = ['EVI', 'NIR', 'NBR', 'SWIR1']
sent_redband       = ['RE1', 'RE2', 'RE3']
sent_interaction_terms = [c for c in X_sent_t.columns if c.startswith('height_x_')]

# Curated interactions matching curated spectral sets
sent_top_spectral_interactions = [f'height_x_{b}' for b in sent_top_spectral
                                  if f'height_x_{b}' in sentinel_raw_df.columns]
sent_redband_interactions      = [f'height_x_{b}' for b in sent_redband
                                  if f'height_x_{b}' in sentinel_raw_df.columns]

sent_features_list = [
    # Non-spectral baselines
    #struct_features,  # already tested in EMIT
    #species,          # already tested in EMIT
    sent_struct_features + species,

    # Spectral alone
    sent_bands,
    sent_indices,
    sent_top_spectral,
    sent_redband,

    # Spectral + height
    sent_bands        + sent_struct_features,
    sent_indices      + sent_struct_features,
    sent_top_spectral + sent_struct_features,
    sent_redband      + sent_struct_features,

    # Spectral + height + species
    sent_bands        + sent_struct_features + species,
    sent_top_spectral + sent_struct_features + species,

    # Spectral + height + interactions (curated where it matters)
    sent_bands        + sent_struct_features + sent_interaction_terms,
    sent_top_spectral + sent_struct_features + sent_top_spectral_interactions,
    sent_redband      + sent_struct_features + sent_redband_interactions,

    # Full specturm of features
    sent_bands        + sent_indices + sent_struct_features + sent_interaction_terms + species
]

## Sentiel. Create plot groups.

In [15]:
# Retain the groups/plot_id for splitting the data based on groups.
plot_groups_sent = None
if 'plot_id' in X_sent_t.columns:
    plot_groups_sent = X_sent_t['plot_id'].copy()
    X_sent_t = X_sent_t.drop(columns=['plot_id'])

if 'plot_id' in X_sent_s.columns:
    X_sent_s = X_sent_s.drop(columns=['plot_id'])

near_zero_sites_sent, high_agb_sites_sent,\
    near_zero_plots_sent, high_agb_plots_sent = \
        get_low_and_high_agb_plots(y_sent,
                                   plot_groups_sent)

grp_sentinel = GROUP_INFO(near_zero_sites_sent,
                          high_agb_sites_sent,
                          near_zero_plots_sent,
                          high_agb_plots_sent,
                          groups=plot_groups_sent,
                          cv=test_cv)

High-AGB threshold  : 1336.17 kg
Near-zero threshold : 0.000052

Near-zero variance plots:
  Big Creek_1               : log var = 0.000036
  Big Creek_4               : log var = 0.000032

High-AGB plots:
  ACA_Acarau Boca           : max AGB = 1468.1 kg
  Arco_del_Espino_18_1      : max AGB = 1564.3 kg
  Arco_del_Espino_18_3      : max AGB = 2403.6 kg
  Arco_del_Espino_18_6      : max AGB = 2403.6 kg
  BAR_Barreto               : max AGB = 2169.6 kg
  BOC_Boca Grande           : max AGB = 5572.3 kg
  Bordo_del_Chile_23_2      : max AGB = 1542.6 kg
  CAE_Caetano               : max AGB = 7364.1 kg
  Desembocadura_Río_Lempa_1_2 : max AGB = 2520.3 kg
  Desembocadura_Río_Lempa_1_3 : max AGB = 1641.7 kg
  Desembocadura_Río_Lempa_1_4 : max AGB = 1594.4 kg
  Desembocadura_Río_Lempa_1_6 : max AGB = 2739.9 kg
  El R’o_5                  : max AGB = 1371.5 kg
  El R’o_6                  : max AGB = 1421.6 kg
  El_Aguacate_24_1          : max AGB = 1627.4 kg
  El_Plan_de_la_Ceiba_8_1   : max AG

In [16]:
assert plot_groups_sent is not None

# SENTINEL-2 EXPERIMENTS

In [17]:
test_cv = 5

In [18]:
LINEAR_FULL    = ["linear", "ridge", "lasso", "elasticnet"]
LINEAR_NO_OLS  = ["ridge", "lasso", "elasticnet"]

data_combos= [("SENTINEL + TANDEMX", X_sent_t, y_sent, sent_features_list, grp_sentinel, LINEAR_FULL),
              ("SENTINEL + SIMARD" , X_sent_s, y_sent, sent_features_list, grp_sentinel, LINEAR_FULL)
             ]
model_list = ["linear", "rf", "xgboost", "lgbm", "merf"]
is_grids   = [False, True]
is_groups  = [True]

In [ ]:
global_experiment_list = {}
experiments_by_category = {}

for (combo_name, X, y, features_list, grp, linear_variants), model, use_grid, use_groups in product(
    data_combos, model_list, is_grids, is_groups
):
    print(f"\n{'='*70}")
    print(f"DATA: {combo_name} | MODEL: {model} | GRID: {use_grid} | GROUPED: {use_groups}")
    print('='*70)

    #run_label = f"{combo_name}, {model}, grid_{'yes' if use_grid else 'no'}, GROUPED: {'yes' if use_groups else 'no'}"

    exp_result = run_experiment(
        X, y,
        is_groups       = use_groups,
        group_info      = grp,
        features_list   = features_list,
        model_type      = model,
        linear_variants = linear_variants,
        is_grid         = use_grid,
        is_stratified   = False,
        exp_total_list = global_experiment_list,
        exp_by_category=experiments_by_category,
        label           = combo_name
    )


DATA: SENTINEL + TANDEMX | MODEL: linear | GRID: False | GROUPED: True

[1/60]

 EXPERIMENT-1. SENTINEL + TANDEMX,Model: LINEAR REGRESSION, Grouping? Yes, Hypertuned? No, Features: ['height', 'species']
Test R²     : -0.0187
Test RMSE   : 386.99 kg
Train R² (log scale): 0.3602
Train R² (orig scale): 0.0082
Train RMSE  : 367.56 kg
Num rows    : 7123
Num Features: 19

 Cross-validation ---
CV R² mean: -0.8526
CV R² std : 0.6938
CV scores : [-2.149 -0.994 -0.448 -0.395 -0.277]

Grouped Cross-validation ---
Grouped CV R² mean: 0.3090
Grouped CV R² std : 0.1432
Grouped CV scores : [0.347 0.303 0.569 0.134 0.236 0.401 0.432 0.034 0.299 0.335]
 EVALUATION: EXPERIMENT-1. SENTINEL + TANDEMX,Model: LINEAR REGRESSION, Grouping? Yes, Hypertuned? No, Features: ['height', 'species']

Test set:
  R²   : -0.019
  RMSE : 386.99 kg

 ❌ Test R² is negative (-0.019)

Regular CrossValidation:
  Mean   : -0.853
  Std    : 0.694
  Scores : [-2.149 -0.994 -0.448 -0.395 -0.277]
 ❌ CV mean is negative (-0.853)

# FIND THE BEST EXPERIMENT SO FAR.

## SUMMARY OF EXPERIMENTS

In [ ]:
%run Model_functions.ipynb
best_results = filter_best_experiments(global_experiment_list)
tab_df = tabulate_results(best_results)

## BEST ONE

In [ ]:
%run Model_functions.ipynb
best_result = best_results[0][1]
print_experiment(best_result)